In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    when,
    sha2,
    concat_ws,
    col,
    concat,
    date_format,
    regexp_extract,
    regexp_replace,
    trim,
    initcap,
    to_date,
    months_between,
    current_date,
    floor

)


import uuid
# =====================================================
# PARAMETERS
# =====================================================

dbutils.widgets.text("source_table", "")
source_table = dbutils.widgets.get("source_table")

pipeline_run_id = str(uuid.uuid4())

print(f"Processing: {source_table}")
# =====================================================
# CONFIG
# =====================================================

silver_catalog = "silver"

# source_table example:


table_name = source_table.split(".")[-1]

target_table = f"{silver_catalog}.{table_name}"

watermark_column = "migrated_at"

# =====================================================
# CREATE SCHEMA
# =====================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}")

# =====================================================
# CREATE HASH
# =====================================================
metadata_columns = [
    "migrated_at",
    "ingestion_date",
    "batch_id",
    "source_table",
    "source_system",
    "load_start_id",
    "load_end_id",
    "pipeline_run_id",
    "uniq_id"
]


# =====================================================
# READ SOURCE
# =====================================================

source_df = spark.table(source_table)
print('THE SOURCE TABLE',source_table)

# =====================================================
# Customers Tranformation
# =====================================================
if table_name == "customers":
    source_df = source_df.withColumns(
        {
        "first_name": trim(regexp_replace(col("first_name"),"_","")),
        "last_name": trim(regexp_replace(col("last_name"),"_","")),
        "occupation": trim(regexp_replace(col("occupation"),"_","")),
        "ssn": trim(regexp_replace(col("ssn"),"[^A-Za-z0-9]","")),
        "phone": concat(lit("+1"),col("phone")),
        "dob": date_format(col("dob"),"dd-MMM-yyyy"),
        "age": floor(months_between(current_date(),to_date(col("dob"), "dd-MMM-yyyy")) / 12).cast("int")
    })  
    print('Customers Dataframe in Transforamtion')
    source_df.show(5,truncate=False)
    
    

# =====================================================
# Address Tranformation
# =====================================================
if table_name == "addresses":
    source_df = source_df.withColumns(

        {
            "state" : col("city"),
            "city" : when(col("city") == "CA", "Los Angeles")
            .when(col("city") == "WA", "Seattle")
            .when(col("city") == "FL", "Miami")
            .when(col("city") == "NY", "New York")
            .when(col("city") == "TX", "Dallas")
            .otherwise("Unknown")
        }
    )
    print('Address Dataframe in Transforamtion')
    source_df.show(2,truncate=False)
    

# =====================================================
# Application Tranformation
# =====================================================
if table_name == "applications":
    source_df = source_df.withColumns(
        {
         "customer_type": when(col("coverage_amount")>=150000,"Premium_customer").otherwise("General_customer")
        }
    )
    
    print('Applications Dataframe in Transforamtion')
    source_df.show(2,truncate=False)
                                                        

# # =====================================================
# Audit Log Tranformation
# =======================================================
if table_name == "audit_log":
    source_df = (
    source_df
    .withColumns({"payload": regexp_replace(col("payload"),r"(AUDIT_?)+","PAYLOAD")})
    .withColumnRenamed("audit_id","audit_log_id")
    .drop("record_id")
    )

    print('Audit_log Dataframe in Transforamtion')
    source_df.show(2,truncate=False)
    

# # =====================================================
# Beneficiaries Tranformation
# =======================================================
if table_name == "beneficiaries":
    source_df = source_df.withColumns(
    {
        "name": regexp_replace(col("name"),"_",""),
        "beneficiary_share": lit("100%")
    }
    )
   
    print('Beneficiaries Dataframe in Transforamtion')
    source_df.show(2,truncate=False)
    

# # =====================================================
# Billing account Tranformation
# =======================================================
if table_name == "billing_accounts":
    source_df = source_df.withColumns(
        {
            "billing_method": when((col("policy_id") % 2) != 0, "ACH").otherwise(col("billing_method")),
        }
        )

    print('Billing_accounts Dataframe in Transforamtion')
    source_df.show(2,truncate=False)
    



# # =====================================================
# Claims Tranformation
# =======================================================
"""NO TRANSFORMATION"""


# # =====================================================
# Agents Tranformation
# =======================================================
if table_name == "agents":
    source_df = source_df.withColumns(
        {
        "license_number":   regexp_replace("license_number","_",""),
        "agent_name":       regexp_replace("agent_name","_",""),
        "agency_name":      regexp_replace("agency_name","_","")
        }   
    )
    
    print('Agents Dataframe after Transforamtion')
    source_df.show(2,truncate=False)
    
# # =====================================================
# Documents Tranformation
# =======================================================
if table_name == "documents":
    source_df = source_df.withColumns(
        {
        "file_blob": concat(lit("XX_"),col("policy_id"),lit("."),col("doc_type")),
        "doc_type":  when(col("document_id") %2==0 ,"Credit Score").otherwise("Medical Records")
        }
    )
    print('Documents Dataframe after Transforamtion')
    source_df.show(2,truncate=False)
    



# # =====================================================
#  Medical_records Tranformation
# =======================================================
if table_name == "medical_records":
    source_df = source_df.withColumns({
        "diagnosis":when(col("medical_record_id") %2==0 ,"Diabetes").otherwise("Heart Issue")
    })

    print('Medical records Dataframe after Transforamtion')
    source_df.show(2,truncate=False)
    
# # =====================================================
# Policies Tranformation
# =======================================================
if table_name == "policies":
    source_df = (
        source_df
        .withColumn("policy_number",regexp_replace("policy_number","_",""))
        
    )
    print('Policies Dataframe after Transforamtion')
    source_df.show(2,truncate=False)
    

# # =====================================================
# Policy dividends Tranformation
# =======================================================
"""NO TRANSFORMATION"""

# # =====================================================
# Policy Riders	 Tranformation
# =======================================================
"""NO TRANSFORMATION"""

# # =====================================================
# Premium Payments Tranformation
# =======================================================
if table_name == "premium_payments":
    source_df = source_df.withColumns({
            "txn" :regexp_replace(col("txn"),r"(TXN_(\d+)_?)+","TXN$2")
    }
    )
    
# # =====================================================
# Billing account Tranformation
# =======================================================
if table_name == "underwriting_assessments":
    source_df = source_df.withColumns({
        "smoker"   : when(col("smoker") == 1,"TRUE").otherwise("False"),
        "notes"    : regexp_replace("notes", r"(OK_?)+", "Decision is Approved"),
        "bmi_type" : when(col("bmi") >= 30,lit("Obesity"))
        .when(col("bmi") >= 25,lit("Overweight"))
        .when(col("bmi") >= 18.5,lit("Normal"))
    }
    )
    print('Underwriting Assessments Dataframe after Transforamtion')
    source_df.show(2,truncate=False)
    
print("Before Bussiness_Columns")
business_columns = [
    c for c in source_df.columns
    if c not in metadata_columns
]

print("Bussiness_Columns",business_columns)

silver_df  = (
        source_df
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .withColumn("silver_ingested_at", current_timestamp())
        .withColumn("pipeline_run_id", lit(pipeline_run_id))
        .withColumn("is_active", lit(True))
        .withColumn(
        "record_hash",
        sha2(
            concat_ws("||", *business_columns),
            256
        )
    )
    )
print("Silver DF IS")
silver_df.show(2, truncate=False)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DecimalType,
    DateType,
    TimestampType,
    BooleanType,
    ShortType
)

# =========================================================
# COMMON METADATA COLUMNS
# =========================================================

metadata_schema = [
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("silver_ingested_at", TimestampType(), True),
    StructField("pipeline_run_id", StringType(), True),
    StructField("is_active", BooleanType(), True),

    StructField("migrated_at", TimestampType(), True),
    StructField("ingestion_date", DateType(), True),
    StructField("batch_id", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("load_start_id", LongType(), True),
    StructField("load_end_id", LongType(), True),
    StructField("uniq_id", LongType(), True),
    StructField("record_hash", StringType(), True),
]

# =========================================================
# silver.customers
# =========================================================

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("age", IntegerType(), False),
    StructField("ssn", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("income", DecimalType(12, 2), True),
    StructField("occupation", StringType(), True),
    

    *metadata_schema
])

# =========================================================
# silver.addresses
# =========================================================

addresses_schema = StructType([
    StructField("address_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("address_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.agents
# =========================================================

agents_schema = StructType([
    StructField("agent_id", IntegerType(), False),
    StructField("agent_name", StringType(), True),
    StructField("license_number", StringType(), True),
    StructField("agency_name", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.applications
# =========================================================

applications_schema = StructType([
    StructField("application_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("agent_id", IntegerType(), True),
    StructField("application_type", StringType(), True),
    StructField("coverage_amount", DecimalType(15, 2), True),
    StructField("status", StringType(), True),
    StructField("customer_type", StringType(), True),
    

    *metadata_schema
])


# =========================================================
# silver.underwriting_assessments
# =========================================================

underwriting_assessments_schema = StructType([
    StructField("assessment_id", IntegerType(), False),
    StructField("application_id", IntegerType(), True),
    StructField("bmi", DecimalType(5, 2), True),
    StructField("smoker", StringType(), True),#StructField("smoker", ShortType(), True),
    StructField("decision", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("bmi_type", StringType(), True),
    

    *metadata_schema
])

# =========================================================
# silver.medical_records
# =========================================================

medical_records_schema = StructType([
    StructField("medical_record_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("diagnosis", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.policies
# =========================================================

policies_schema = StructType([
    StructField("policy_id", IntegerType(), False),
    StructField("application_id", IntegerType(), True),
    StructField("policy_number", StringType(), True),
    StructField("status", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.beneficiaries
# =========================================================

beneficiaries_schema = StructType([
    StructField("beneficiary_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("relationship", StringType(), True),
    StructField("beneficiary_share", StringType(), True),
    

    *metadata_schema
])

# =========================================================
# silver.policy_riders
# =========================================================

policy_riders_schema = StructType([
    StructField("rider_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("rider_type", StringType(), True),
    StructField("rider_amount", DecimalType(10, 2), True),

    *metadata_schema
])

# =========================================================
# silver.billing_accounts
# =========================================================

billing_accounts_schema = StructType([
    StructField("billing_account_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("billing_method", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.premium_payments
# =========================================================

premium_payments_schema = StructType([
    StructField("payment_id", LongType(), False),
    StructField("billing_account_id", IntegerType(), True),
    StructField("amount", DecimalType(10, 2), True),
    StructField("status", StringType(), True),
    StructField("txn", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.claims
# =========================================================

claims_schema = StructType([
    StructField("claim_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("claim_amount", DecimalType(12, 2), True),
    StructField("status", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.documents
# =========================================================

documents_schema = StructType([
    StructField("document_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("doc_type", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_blob", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.audit_log
# =========================================================

audit_log_schema = StructType([
    StructField("audit_log_id", LongType(), False),
    StructField("table_name", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("payload", StringType(), True),

    *metadata_schema
])

# =========================================================
# silver.policy_dividends
# =========================================================

policy_dividends_schema = StructType([
    StructField("dividend_id", IntegerType(), False),
    StructField("policy_id", IntegerType(), True),
    StructField("amount", DecimalType(12, 2), True),

    *metadata_schema
])
# =========================================================
# CREATE SILVER DATABASE
# =========================================================

spark.sql("CREATE DATABASE IF NOT EXISTS silver")
# =========================================================
# TABLE NAME -> SCHEMA MAPPING
# =========================================================

table_schemas = {
    "customers": customers_schema,
    "addresses": addresses_schema,
    "agents": agents_schema,
    "applications": applications_schema,
    "underwriting_assessments": underwriting_assessments_schema,
    "medical_records": medical_records_schema,
    "policies": policies_schema,
    "beneficiaries": beneficiaries_schema,
    "policy_riders": policy_riders_schema,
    "billing_accounts": billing_accounts_schema,
    "premium_payments": premium_payments_schema,
    "claims": claims_schema,
    "documents": documents_schema,
    "audit_log": audit_log_schema,
    "policy_dividends": policy_dividends_schema,
}




print(f"Creating silver.{table_name}")

df = spark.createDataFrame([], table_schemas[table_name])

print("All silver tables created successfully.")

# =====================================================
# FULL LOAD
# =====================================================
target_schema = table_schemas[table_name]
# =====================================================
# ENFORCE TARGET SCHEMA
# =====================================================

target_columns = [field.name for field in target_schema.fields]
print('target_columns',target_columns)


silver_df = silver_df.select(
    *[
        col(field.name).cast(field.dataType).alias(field.name)
        for field in target_schema.fields
    ]
)
silver_df.show(5)

silver_df = silver_df.dropDuplicates(["record_hash"])
# =====================================================
# WRITE TO SILVER
# =====================================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Loaded data into {target_table}")